In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- common_get_observations ---
def _prepare_x_axis(values):
    return list(values)

class _FakeArray:
    def __init__(self, values):
        self.values = np.asarray(values)

class _FakeDataset(dict):
    def __init__(self, observations, std, axis_name="index", axis_values=(0, 1)):
        super().__init__({"observations": _FakeArray(observations), "std": _FakeArray(std), axis_name: _FakeArray(axis_values)})
        self.coords = {axis_name: self[axis_name]}

def make_common_get_observations_experiment():
    return SimpleNamespace(name="test", params={}, observations={
        "obs_b": _FakeDataset([[2.0, 3.0]], [[0.2, 0.3]], axis_values=(2, 3)),
        "obs_a": _FakeDataset([[1.0]], [[0.1]], axis_values=(1,)),
    })

def make_common_get_observations_list():
    return []

FIX_COMMON_GET_OBSERVATIONS_EXPERIMENT = make_common_get_observations_experiment()
FIX_COMMON_GET_OBSERVATIONS_OBSERVATIONS = make_common_get_observations_list()

# --- common_summary_filter ---
FIX_COMMON_SUMMARY_FILTER_KEY = "mAP"
_summary_mi = pd.MultiIndex.from_tuples([(1,"mAP",1),(2,"mAP",2)], names=["realization","name","time"])
_summary_df = pd.DataFrame({"value":[0.8,0.9]}, index=_summary_mi)
FIX_COMMON_SUMMARY_FILTER_SUMMARY_DATA = SimpleNamespace(to_dataframe=lambda: _summary_df)

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_common_get_observations(experiment, observations):
    for key, dataset in experiment.observations.items():
        observation = {
            "name": key,
            "values": list(dataset["observations"].values.flatten()),
            "errors": list(dataset["std"].values.flatten()),
        }
        if "time" in dataset.coords:
            observation["x_axis"] = _prepare_x_axis(dataset["time"].values.flatten())
        else:
            observation["x_axis"] = _prepare_x_axis(dataset["index"].values.flatten())
        observations.append(observation)
    observations.sort(key=lambda x: x["x_axis"])
    return None

def before_common_summary_filter(key, summary_data):
    df = summary_data.to_dataframe()
    df = df.xs(key, level="name")
    df.index = df.index.rename(
        {"time": "Date", "realization": "Realization"}
    ).reorder_levels(["Realization", "Date"])
    return df

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_common_get_observations(experiment, observations):
    for key, dataset in experiment.observations.items():
        observation = {
            "name": key,
            "values": list(dataset["observations"].values.flatten()),
            "errors": list(dataset["std"].values.flatten()),
        }
        if "time" in dataset.coords:
            observation["x_axis"] = _prepare_x_axis(dataset["time"].values.flatten())
        else:
            observation["x_axis"] = _prepare_x_axis(dataset["index"].values.flatten())
        observations.append(observation)
    observations.sort(key=lambda x: x["x_axis"])
    return None

def gen_common_summary_filter(key, summary_data):

    df = pl.from_pandas(summary_data.to_dataframe().reset_index())
    df = df.filter(pl.col("name") == key).drop("name")
    df = df.rename({"time": "Date", "realization": "Realization"})
    df = df.select(["Realization", "Date", *[c for c in df.columns if c not in {"Realization", "Date"}]])
    return df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: common_summary_filter ===

# L1 smoke – generated
try:
    _r = gen_common_summary_filter(FIX_COMMON_SUMMARY_FILTER_KEY, FIX_COMMON_SUMMARY_FILTER_SUMMARY_DATA)
    print("✅ L1 smoke gen_common_summary_filter: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_common_summary_filter: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_common_summary_filter(FIX_COMMON_SUMMARY_FILTER_KEY, FIX_COMMON_SUMMARY_FILTER_SUMMARY_DATA)
    print("✅ L1 smoke before_common_summary_filter: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_common_summary_filter: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_common_summary_filter(FIX_COMMON_SUMMARY_FILTER_KEY, FIX_COMMON_SUMMARY_FILTER_SUMMARY_DATA)
    _rg = gen_common_summary_filter(FIX_COMMON_SUMMARY_FILTER_KEY, FIX_COMMON_SUMMARY_FILTER_SUMMARY_DATA)
    compare(_rb, _rg, "common_summary_filter")
except Exception as _e:
    print(f"❌ L2 equivalence common_summary_filter: setup error — {type(_e).__name__}: {_e}")

# L3 edge - a missing summary key compares exception behavior.
try:
    _before_err = _gen_err = None
    _before_result = _gen_result = None
    try:
        _before_result = before_common_summary_filter("__missing__", FIX_COMMON_SUMMARY_FILTER_SUMMARY_DATA)
    except Exception as _e:
        _before_err = _e
    try:
        _gen_result = gen_common_summary_filter("__missing__", FIX_COMMON_SUMMARY_FILTER_SUMMARY_DATA)
    except Exception as _e:
        _gen_err = _e
    if isinstance(_before_err, KeyError) and _gen_err is not None and not isinstance(_gen_err, (SyntaxError, NameError)):
        print(f"✅ L3 edge common_summary_filter missing key: MATCH - both rejected (before={type(_before_err).__name__}, gen={type(_gen_err).__name__})")
    elif _before_err is None and _gen_err is None:
        compare(
            _before_result, _gen_result,
            "L3 edge common_summary_filter missing key",
            check_row_order=True,
        )
    else:
        print(
            "❌ L3 edge common_summary_filter missing key: MISMATCH - "
            f"before_error={type(_before_err).__name__ if _before_err else None}, "
            f"gen_error={type(_gen_err).__name__ if _gen_err else None}"
        )
except Exception as _e:
    print(f"❌ L3 edge common_summary_filter missing key: {type(_e).__name__}: {_e}")
